In [ ]:
# Esame 654AA - a.a. 2025/2026
# Studenti: Leonardo Celati, Samuele Taviano
# Matricole: 660185, ?

<h1>CUP Datasets SVM</h1>
<p>Exploring the cup dataset with SVM Classifier.</p>
<hr/>

In [ ]:
import importlib

import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, mean_absolute_error, mean_squared_error, \
    r2_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, cross_val_predict, KFold
import cup_common as cc
import svm_common as sc

In [ ]:
importlib.reload(sc)
importlib.reload(cc)

<h2>Exploring Datasets</h2>
<hr/>

In [ ]:
# Note: The regularization parameter C is explored on a logarithmic scale over four orders of magnitude,
# from strong (0.1) to weak regularization (100), following standard practice for SVM hyperparameter tuning.
param_grid_default = {
    "svr__estimator__kernel": ["rbf", "linear", "poly", "sigmoid"],
    "svr__estimator__C": [0.1, 1, 10, 100],
    "svr__estimator__gamma": ["scale", 0.1, 0.01, 0.001],
    "svr__estimator__epsilon": [0.01, 0.05, 0.1, 0.2],
    "svr__estimator__degree": [2, 3, 4],
}

# We only test SVC
pipe_default = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("svr", MultiOutputRegressor(SVR()))
])

scoring="neg_mean_squared_error"

# CV params
n_split = 5
cv_default = KFold(n_splits=n_split, shuffle=True, random_state=42)


<h3>Monk 1</h3>

<h4>Data Loading</h4>
<p>Load and prepare data</p>

In [ ]:
df_train, df_test = cc.load_set()
X_tr, y_tr, X_ts, y_ts = cc.split_and_prepare_dataset(df_train, ratio=0.2)
features_names = X_tr.columns
cc.dataset_introspection(df_train, df_test)

<h4>Model Selection</h4>
<p>Finding the best model with GridSearchCV.</p>

In [ ]:
# Perform a grid search
grid = GridSearchCV(
    estimator = pipe_default,
    param_grid = param_grid_default,
    cv = cv_default,
    scoring = scoring,
    n_jobs = -1 # use all available cores
)

# Fit TR features and labels into grid
grid.fit(X_tr, y_tr)
best_model = grid.best_estimator_
sc.grid_introspection(grid)

<h4>Model Assesment</h4>
<p>Collecting various metrics for model assesment.</p>

<h5>Cross-validated prediction</h5>
<p>Cross-validated predictions are computed using a k-fold cross-validation strategy, where the dataset is partitioned into multiple folds and, at each iteration, the model is trained on the training folds and used to predict the held-out fold.
This procedure ensures that each prediction is generated by a model that has not been trained on the corresponding sample, providing an unbiased estimate of the model’s generalization behavior on unseen data.</p>

In [ ]:
cv_predictions = cross_val_predict(
    best_model,
    X_tr, y_tr,
    cv=cv_default
)

#print(classification_report(monk_1_TR_labels, cv_predictions_1, zero_division=0))

In [ ]:
mae = mean_absolute_error(y_tr, cv_predictions)
mse = mean_squared_error(y_tr, cv_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_tr, cv_predictions)

print(f"MAE  = {mae:.3f}")
print(f"MSE  = {mse:.3f}")
print(f"RMSE = {rmse:.3f}")
print(f"R²   = {r2:.3f}")

<h5>Cross Validate Predictions Plot</h5>
<p>
This plot compares the true target values with the corresponding predictions obtained through cross-validation.
Each point represents a data sample, while the dashed diagonal line indicates the ideal case in which predicted values perfectly match the true values.
The dispersion of points around the diagonal provides insight into the model’s prediction accuracy, while systematic deviations from the ideal line highlight potential biases or limitations in capturing the underlying relationship between input features and target values.
A wide spread or compression of predictions toward the center suggests reduced accuracy for extreme target values.
</p>

In [ ]:
sc.plot_true_vs_preds_svr(y_tr, cv_predictions)

<h5>Learning Curves</h5>
<p>This plot shows the evolution of training and cross-validation errors as the training set size increases. The learning curve helps assess whether the model suffers from underfitting or overfitting and whether performance is limited by model capacity or by intrinsic noise in the data.</p>

In [ ]:
sc.plot_learning_curve_svr(best_model, X_tr, y_tr, cv_default, scoring=grid.scoring)

<h4>Test predictions</h4>
<p>Test predictions on the test set obtained from the train set</p>

In [ ]:
y_pred = best_model.predict(X_ts)

In [ ]:
sc.plot_true_vs_preds_svr(y_ts, y_pred)

<h5>Residuals</h5>
<p>This plot shows the distribution of the residuals, defined as the difference between the true target values and the corresponding model predictions.
A residual distribution centered around zero indicates the absence of a systematic global bias in the model predictions.
The spread and shape of the distribution provide insight into the magnitude and variability of the prediction errors, as well as the presence of large errors or outliers.
Deviations from a narrow, symmetric distribution suggest limitations in the model’s ability to accurately capture the underlying relationship between input features and target values.</p>

In [ ]:
sc.plot_residuals_hist(y_ts, y_pred)

<hr/>